# Company U Genre Classifier

Classify books from Company U library using OpenLibrary subjects and AI-powered zero-shot classification.

In [11]:
import sys
import os
import json
import re
import unicodedata
from typing import List, Tuple, Dict, Optional
from collections import defaultdict
from functools import lru_cache
import concurrent.futures

import pandas as pd
import langid
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    from tqdm import tqdm
except Exception:
    tqdm = lambda x, **k: x

import warnings
warnings.filterwarnings("ignore", category=Warning)

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Configuration

In [12]:
# =========================================================
# CONFIG
# =========================================================
COMPANY_U_INPUT_FILE = "company_u_books_output.csv"
OUTPUT_COMPANY_U = "company_u_genres_output.csv"
TOP_K_GENRES = 3
OPENLIBRARY_CACHE_FILE = "openlibrary_company_u_cache.json"

print(f"Input file: {COMPANY_U_INPUT_FILE}")
print(f"Output file: {OUTPUT_COMPANY_U}")
print(f"Top genres per book: {TOP_K_GENRES}")

Input file: company_u_books_output.csv
Output file: company_u_genres_output.csv
Top genres per book: 3


## Language Detection

In [13]:
LANGUAGE_NAMES = {
    "en": "English",
    "fr": "French",
    "es": "Spanish",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "ru": "Russian",
    "hy": "Armenian",
    "tr": "Turkish",
    "ar": "Arabic",
    "zh": "Chinese",
    "ja": "Japanese",
    "ko": "Korean",
}

EN_HINT_WORDS = frozenset({"the", "a", "an", "of", "and", "to", "in", "for", "with", "on"})

@lru_cache(maxsize=1024)
def detect_language(title: str, min_confidence_latin: float = 0.35) -> Tuple[str, str, float]:
    text = (title or "").strip()
    if not text:
        return "unknown", "Unknown", 0.0

    # quick-script heuristics
    for ch in text:
        cp = ord(ch)
        if 0x0530 <= cp <= 0x058F:
            return "hy", "Armenian", 1.0
        if 0x0400 <= cp <= 0x04FF:
            return "cyr", "Cyrillic", 1.0
        if 0x0600 <= cp <= 0x06FF:
            return "ar", "Arabic", 1.0
        if 0x0590 <= cp <= 0x05FF:
            return "he", "Hebrew", 1.0
        if 0x0370 <= cp <= 0x03FF:
            return "el", "Greek", 1.0
        if 0x4E00 <= cp <= 0x9FFF:
            return "cjk", "CJK", 1.0

    normalized = unicodedata.normalize("NFKD", text)
    normalized = "".join(c for c in normalized if not unicodedata.combining(c))

    words = {w.lower() for w in normalized.replace("'", " ").split()}
    if words & EN_HINT_WORDS:
        return "en", "English", 0.99

    code, conf = langid.classify(normalized)
    if conf >= min_confidence_latin:
        return code, LANGUAGE_NAMES.get(code, code), float(conf)

    return "latin", "Latin (Unknown language)", float(conf)

print("✅ detect_language() function defined")

✅ detect_language() function defined


## Genre Mapping

In [14]:
GENRES = [
    "Fantasy",
    "Science Fiction",
    "Romance",
    "Mystery",
    "Thriller",
    "Historical Fiction",
    "Nonfiction",
    "Biography",
    "Young Adult",
    "Horror",
]

GENRE_KEYWORDS = {
    "Fantasy": frozenset(["fantasy", "magic", "dragon", "myth", "middle earth"]),
    "Science Fiction": frozenset(["science fiction", "sci-fi", "space", "alien", "dystop"]),
    "Romance": frozenset(["romance", "love"]),
    "Mystery": frozenset(["mystery", "detective", "crime"]),
    "Thriller": frozenset(["thriller", "suspense"]),
    "Horror": frozenset(["horror", "ghost", "haunted"]),
    "Biography": frozenset(["biography", "autobiography", "memoir"]),
    "Nonfiction": frozenset(["nonfiction", "history", "business", "psychology", "self-help"]),
    "Historical Fiction": frozenset(["historical fiction"]),
    "Young Adult": frozenset(["young adult", "ya"]),
}

WHITESPACE_REGEX = re.compile(r"\s+")

def normalize(s: str) -> str:
    return WHITESPACE_REGEX.sub(" ", (s or "").lower().strip())

def map_subjects_to_genres(subjects: List[str], top_k: int) -> List[str]:
    if not subjects:
        return []
    text = " | ".join(normalize(x) for x in subjects)
    found = []
    for genre, keys in GENRE_KEYWORDS.items():
        if any(k in text for k in keys):
            found.append(genre)
            if len(found) >= top_k:
                break
    return found[:top_k]

print(f"✅ {len(GENRES)} genres defined")

✅ 10 genres defined


## OpenLibrary API with Caching & Retries

In [15]:
OPENLIBRARY_CACHE: Dict[str, List[str]] = {}

def _load_cache():
    try:
        if os.path.exists(OPENLIBRARY_CACHE_FILE):
            with open(OPENLIBRARY_CACHE_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, dict):
                    OPENLIBRARY_CACHE.update(data)
    except Exception:
        pass

def _save_cache():
    try:
        with open(OPENLIBRARY_CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(OPENLIBRARY_CACHE, f, ensure_ascii=False)
    except Exception:
        pass

_session: Optional[requests.Session] = None

def _session_with_retries():
    global _session
    if _session is None:
        s = requests.Session()
        retries = Retry(total=3, backoff_factor=0.6, status_forcelist=(500,502,503,504))
        s.mount("https://", HTTPAdapter(max_retries=retries))
        _session = s
    return _session

_load_cache()

def openlibrary_get_subjects(title: str) -> List[str]:
    """Disk-backed cached lookup with a shared session and retries."""
    title = (title or "").strip()
    if not title:
        return []
    if title in OPENLIBRARY_CACHE:
        return OPENLIBRARY_CACHE[title]

    session = _session_with_retries()
    try:
        r = session.get("https://openlibrary.org/search.json", params={"title": title}, timeout=8)
        r.raise_for_status()
        data = r.json()
        docs = data.get("docs", [])
        if docs and docs[0].get("subject"):
            subjects = docs[0]["subject"]
            OPENLIBRARY_CACHE[title] = subjects
            return subjects
        if docs:
            work_key = docs[0].get("key")
            if work_key:
                w = session.get(f"https://openlibrary.org{work_key}.json", timeout=8)
                w.raise_for_status()
                subjects = w.json().get("subjects", []) or []
                OPENLIBRARY_CACHE[title] = subjects
                return subjects
    except Exception:
        OPENLIBRARY_CACHE[title] = []
        return []
    OPENLIBRARY_CACHE[title] = []
    return []

print("✅ OpenLibrary functions defined")

✅ OpenLibrary functions defined


## AI Zero-Shot Classifier (Lazy Loading)

In [16]:
_classifier = None
_classifier_model = "valhalla/distilbart-mnli-12-1"

def get_classifier():
    global _classifier
    if _classifier is None:
        try:
            import torch
            from transformers import pipeline, AutoConfig
            device = 0 if torch.cuda.is_available() else -1
            cfg = AutoConfig.from_pretrained(_classifier_model)
            cfg.tie_word_embeddings = False
            _classifier = pipeline("zero-shot-classification", model=_classifier_model, config=cfg, device=device)
        except Exception:
            _classifier = None
    return _classifier

def get_genres_for_titles(titles: List[str], top_k: int) -> List[List[str]]:
    # try subjects first, then batch classify missing
    mapped: List[List[str]] = [map_subjects_to_genres(openlibrary_get_subjects(t), top_k) for t in titles]
    missing_idx = [i for i, m in enumerate(mapped) if not m]
    if not missing_idx:
        return mapped

    clf = get_classifier()
    if clf is None:
        return mapped

    batch_size = 16
    for i in range(0, len(missing_idx), batch_size):
        batch_idx = missing_idx[i:i+batch_size]
        batch_titles = [titles[j] for j in batch_idx]
        try:
            res = clf(batch_titles, candidate_labels=GENRES, hypothesis_template="This book is a {} book.")
        except Exception:
            res = []
        if isinstance(res, dict):
            res = [res]
        for j, r in enumerate(res):
            labels = r.get("labels", [])[:top_k]
            mapped[batch_idx[j]] = labels
    return mapped

print("✅ AI classifier functions defined")

✅ AI classifier functions defined


## Data Preparation

In [17]:
def prepare_company_u(path: str) -> pd.DataFrame:
    """Load Company U books output file, skip SUMMARY row."""
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig", on_bad_lines="skip",
                     dtype={"Title": "string", "Number": "string"})
    # Remove SUMMARY row
    df = df[df["Title"] != "SUMMARY"].copy()
    df["Title"] = df["Title"].astype(str).str.strip()
    df["Number"] = pd.to_numeric(df["Number"], errors="coerce").fillna(0)
    return df.reset_index(drop=True)

print("✅ prepare_company_u() function defined")

✅ prepare_company_u() function defined


## Load Input Data

In [18]:
df_in = prepare_company_u(COMPANY_U_INPUT_FILE)
print(f"Loaded {len(df_in)} books from {COMPANY_U_INPUT_FILE}")
print(f"\nFirst 5 books:")
print(df_in.head())

Loaded 666 books from company_u_books_output.csv

First 5 books:
                                               Title  Number
0                            the bastard of istanbul       3
1  business model generation a handbook for visio...       3
2                                    college algebra       3
3                                 kafka on the shore       3
4        the norton anthology of american literature       3


## Core Pipeline

In [19]:
def run_pipeline(df: pd.DataFrame, output_file: str):
    titles = df["Title"].astype(str).tolist()
    numbers = df["Number"].astype(float).tolist()

    # fast language detection
    print("\n1. Detecting languages...")
    langs = [detect_language(t)[1] for t in titles]
    print(f"   ✅ Detected {len(set(langs))} unique languages")

    # parallel OpenLibrary lookups
    print("\n2. Looking up subjects on OpenLibrary (parallel, 8 workers)...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        subjects_list = list(ex.map(openlibrary_get_subjects, titles))
    found_count = sum(1 for s in subjects_list if s)
    print(f"   ✅ Found subjects for {found_count}/{len(titles)} books")

    # map subjects to genres
    print("\n3. Mapping subjects to genres...")
    mapped = [map_subjects_to_genres(s, TOP_K_GENRES) for s in subjects_list]
    mapped_count = sum(1 for m in mapped if m)
    print(f"   ✅ Mapped {mapped_count}/{len(titles)} books to genres")

    # batch-classify missing
    missing = [i for i, m in enumerate(mapped) if not m]
    if missing:
        print(f"\n4. AI zero-shot classification (batched, batch_size=16) for {len(missing)} remaining books...")
        to_classify = [titles[i] for i in missing]
        classified = get_genres_for_titles(to_classify, TOP_K_GENRES)
        for idx, labels in zip(missing, classified):
            mapped[idx] = labels
        print(f"   ✅ Classified {len(missing)} books with AI")

    rows = []
    genre_title_count = defaultdict(int)
    genre_number_sum = defaultdict(float)

    for title, num, lang, genres in zip(titles, numbers, langs, mapped):
        genres = genres or []
        for g in genres:
            genre_title_count[g] += 1
            genre_number_sum[g] += num / max(len(genres), 1)
        rows.append({"Title": title, "Number": num, "language": lang, "Genres": ", ".join(genres)})

    result_df = pd.DataFrame(rows)

    top_titles = sorted(genre_title_count.items(), key=lambda x: x[1], reverse=True)[:5]
    top_numbers = sorted(genre_number_sum.items(), key=lambda x: x[1], reverse=True)[:5]
    summary_text = (f"Top genres by title count: {top_titles}. "
                    f"Top genres by total Number: {[(g, int(v)) for g, v in top_numbers]}.")

    output_data = rows + [{"Title": "", "Number": "", "language": "", "Genres": ""},
                         {"Title": "SUMMARY", "Number": int(result_df["Number"].sum()), "language": "", "Genres": summary_text}]

    final_df = pd.DataFrame(output_data)
    final_df.to_csv(output_file, index=False, encoding="utf-8")

    _save_cache()
    print(f"\n✅ Saved {output_file} ({len(rows)} titles processed)")
    return final_df

print("✅ run_pipeline() function defined")

✅ run_pipeline() function defined


## Run Classification Pipeline

In [20]:
print(f"\n{'='*60}")
print("COMPANY U GENRE CLASSIFICATION PIPELINE")
print(f"{'='*60}")
result_df = run_pipeline(df_in, OUTPUT_COMPANY_U)
print(f"{'='*60}")


COMPANY U GENRE CLASSIFICATION PIPELINE

1. Detecting languages...
   ✅ Detected 8 unique languages

2. Looking up subjects on OpenLibrary (parallel, 8 workers)...
   ✅ Found subjects for 308/666 books

3. Mapping subjects to genres...
   ✅ Mapped 115/666 books to genres

4. AI zero-shot classification (batched, batch_size=16) for 551 remaining books...


Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


   ✅ Classified 551 books with AI

✅ Saved company_u_genres_output.csv (666 titles processed)


## Results Summary

In [21]:
# Remove SUMMARY row for stats
results = result_df[result_df['Title'] != 'SUMMARY'].copy()
results = results[results['Title'] != ''].copy()

print("\n" + "="*60)
print("CLASSIFICATION RESULTS")
print("="*60)
print(f"Books processed: {len(results)}")
print(f"Books with genres: {len(results[results['Genres'] != ''])}")
print(f"Total checkouts: {results['Number'].sum():.0f}")
print(f"\nSample results (first 10 books):")
print(results[['Title', 'Number', 'language', 'Genres']].head(10).to_string())
print(f"\n{'='*60}")


CLASSIFICATION RESULTS
Books processed: 666
Books with genres: 666
Total checkouts: 718

Sample results (first 10 books):
                                                                                   Title Number                  language                                   Genres
0                                                                the bastard of istanbul    3.0                   English                Horror, Fantasy, Thriller
1     business model generation a handbook for visionaries game changers and challengers    3.0                   English             Nonfiction, Fantasy, Mystery
2                                                                        college algebra    3.0  Latin (Unknown language)            Nonfiction, Thriller, Mystery
3                                                                     kafka on the shore    3.0                   English            Nonfiction, Fantasy, Thriller
4                                            the norton anthol